In [17]:
# pip install openpyxl

# Graphviz PNG image Creator with json

In [18]:
import pandas as pd
from graphviz import Digraph
import os
import json

# --- Load the dataset ---
df = pd.read_excel("device_compromise_dataset.xlsx", index_col=False)

# Create output directory for images and JSON
output_dir = "attack_tree_images"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

tactic_column = "tactics"
df[tactic_column] = df[tactic_column].str.strip()

# MITRE Enterprise tactic order (adjust if needed)
mitre_tactic_order = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution", "Persistence", 
    "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery", 
    "Lateral Movement", "Collection", "Command and Control", "Exfiltration", "Impact"
]

unique_tactics = df[tactic_column].dropna().unique().tolist()
ordered_tactics = [t for t in mitre_tactic_order if t in unique_tactics]
print(f"Processing {len(ordered_tactics)} tactics in this order: {ordered_tactics}")

# Helper: Extract all requested metadata per node type
def extract_node_metadata(df_subset, node_type):
    def get_col(col): 
        return df_subset[col].iloc[0] if not df_subset.empty and col in df_subset.columns else None

    if node_type == "cwe":
        metadata = {
            'CVE ID': get_col('CVE ID'),
            # 'Description': get_col('Description'),
            'cwe_Name': get_col('cwe_Name'),
            # 'cwe_Abstraction': get_col('cwe_Abstraction'),
            # 'cwe_Structure': get_col('cwe_Structure'),
            # 'cwe_Status': get_col('cwe_Status'),
            # 'cwe_Description': get_col('cwe_Description'),
            # 'cwe_Extended_Description': get_col('cwe_Extended_Description'),
            # 'cwe_Related_Weaknesses': get_col('cwe_Related_Weaknesses'),
            # 'cwe_Weakness_Ordinality': get_col('cwe_Weakness_Ordinality'),
            # 'cwe_Language_Name': get_col('cwe_Language_Name'),
            # 'cwe_Technology_Class': get_col('cwe_Technology_Class'),
            # 'cwe_Alternate_Terms': get_col('cwe_Alternate_Terms'),
            # 'cwe_Alternate_Terms_With_Descriptions': get_col('cwe_Alternate_Terms_With_Descriptions'),
            # 'cwe_Background_Details': get_col('cwe_Background_Details'),
            'cwe_Modes_Of_Introduction': get_col('cwe_Modes_Of_Introduction'),
            'cwe_Likelihood_Of_Exploit': get_col('cwe_Likelihood_Of_Exploit'),
            # 'cwe_Common_Consequences': get_col('cwe_Common_Consequences'),
            # 'cwe_Detection_Methods': get_col('cwe_Detection_Methods'),
            # 'cwe_Potential_Mitigations': get_col('cwe_Potential_Mitigations'),
            # 'cwe_Demonstrative_Examples': get_col('cwe_Demonstrative_Examples'),
            # 'cwe_Observed_Examples': get_col('cwe_Observed_Examples'),
            # Additional common CVSS fields that sometimes apply to CWEs in your dataset
            'attackVector': get_col('attackVector'),
            'baseScore': get_col('baseScore'),
            'baseSeverity': get_col('baseSeverity'),
            'attackComplexity': get_col('attackComplexity'),
            'privilegesRequired': get_col('privilegesRequired'),
            'userInteraction': get_col('userInteraction'),
            'confidentialityImpact': get_col('confidentialityImpact'),
            'availabilityImpact': get_col('availabilityImpact'),
            'exploitabilityScore': get_col('exploitabilityScore'),
        }
    elif node_type == "capec":
        metadata = {
            # 'CVE ID': get_col('CVE ID'),
            'capec_Name': get_col('capec_Name'),
            # 'capec_Mitre_Techniques': get_col('capec_Mitre_Techniques'),
            # 'capec_Abstraction': get_col('capec_Abstraction'),
            # 'capec_Status': get_col('capec_Status'),
            # 'capec_Description': get_col('capec_Description'),
            # 'capec_Extended_Description': get_col('capec_Extended_Description'),
            'capec_Likelihood_Of_Attack': get_col('capec_Likelihood_Of_Attack'),
            'capec_Typical_Severity': get_col('capec_Typical_Severity'),
            # 'capec_Execution_Flow': get_col('capec_Execution_Flow'),
            # 'capec_Prerequisites': get_col('capec_Prerequisites'),
            'capec_Skills_Required': get_col('capec_Skills_Required'),
            # 'capec_Resources_Required': get_col('capec_Resources_Required'),
            # 'capec_Consequences': get_col('capec_Consequences'),
            # 'capec_Mitigations': get_col('capec_Mitigations'),
            # 'capec_Example_Instances': get_col('capec_Example_Instances'),
            # 'capec_Related_Weaknesses': get_col('capec_Related_Weaknesses'),
            # 'capec_Taxonomy_Mappings': get_col('capec_Taxonomy_Mappings'),
        }
    elif node_type == "tech":
        metadata = {
            # 'CVE ID': get_col('CVE ID'),
            # 'Description': get_col('Description'),
            'technique_name': get_col('technique_name'),
            'subtechnique_name': get_col('subtechnique_name'),
            'tactics': get_col('tactics'),
            # 'detection': get_col('detection'),
            # 'platforms': get_col('platforms'),
            # 'data sources': get_col('data sources'),
            # 'defenses bypassed': get_col('defenses bypassed'),
            # 'contributors': get_col('contributors'),
            # 'permissions required': get_col('permissions required'),
            # 'supports remote': get_col('supports remote'),
            # 'system requirements': get_col('system requirements'),
            # 'impact type': get_col('impact type'),
            # 'effective permissions': get_col('effective permissions'),
            # Common CVSS fields for techniques if applicable
            # 'attackVector': get_col('attackVector'),
            # 'baseScore': get_col('baseScore'),
            # 'baseSeverity': get_col('baseSeverity'),
            # 'attackComplexity': get_col('attackComplexity'),
            # 'privilegesRequired': get_col('privilegesRequired'),
            # 'userInteraction': get_col('userInteraction'),
            # 'confidentialityImpact': get_col('confidentialityImpact'),
            # 'availabilityImpact': get_col('availabilityImpact'),
            # 'exploitabilityScore': get_col('exploitabilityScore'),
        }
    elif node_type == "subtech":
        metadata = {
            'subtechnique_name': get_col('subtechnique_name'),
            'technique_name': get_col('technique_name'),
            'tactics': get_col('tactics'),
            # Adding detection and related fields as well if needed
            # 'detection': get_col('detection'),
            # 'platforms': get_col('platforms'),
            # 'data sources': get_col('data sources'),
        }
    else:
        # Fallback or generic with most common metadata
        metadata = {
            # 'CVE ID': get_col('CVE ID'),
            # 'Description': get_col('Description'),
            # 'cwe_Name': get_col('cwe_Name'),
            # 'capec_Name': get_col('capec_Name'),
            # 'technique_name': get_col('technique_name'),
            # 'subtechnique_name': get_col('subtechnique_name'),
            # 'tactics': get_col('tactics'),
            # 'baseScore': get_col('baseScore'),
            # 'baseSeverity': get_col('baseSeverity'),
            # 'attackVector': get_col('attackVector'),
            # 'attackComplexity': get_col('attackComplexity'),
            # 'privilegesRequired': get_col('privilegesRequired'),
            # 'userInteraction': get_col('userInteraction'),
            # 'confidentialityImpact': get_col('confidentialityImpact'),
            # 'availabilityImpact': get_col('availabilityImpact'),
            # 'exploitabilityScore': get_col('exploitabilityScore'),
        }
    return metadata

# Main function to create graph for a tactic
def create_single_tactic_graph(tactic, tactic_index):
    dot = Digraph()
    dot.attr('node', style='filled', shape='box', fontname='Helvetica')

    nodes_json, edges_json = [], []

    root_id = "root"
    tactic_id = f"tactic_{tactic_index}"
    dot.node(root_id, label="Compromise Device", fillcolor="#FFD700")
    dot.node(tactic_id, label=tactic, fillcolor="#98FB98")
    dot.edge(root_id, tactic_id)

    nodes_json += [
        {"id": root_id, "label": "Compromise Device", "type": "root"},
        {"id": tactic_id, "label": tactic, "type": "tactic"}
    ]
    edges_json.append({"from": root_id, "to": tactic_id})

    tactic_df = df[df[tactic_column] == tactic]
    tactic_weaknesses = tactic_df["cwe_Name"].dropna().unique().tolist()
    tactic_weaknesses = [str(cwe).strip() for cwe in tactic_weaknesses if str(cwe).strip()]
    cwe_name_set = set([w.lower() for w in tactic_weaknesses])

    node_counters = {'cwe': 0, 'tech': 0, 'capec': 0, 'subtech': 0, 'gate': 0}
    added_techniques = {}

    def binary_or(parent, children, edges_json):
        if len(children) == 0:
            return
        if len(children) == 1:
            dot.edge(parent, children[0])
            edges_json.append({"from": parent, "to": children[0]})
        elif len(children) == 2:
            or_id = f"or_{tactic_index}_{node_counters['gate']}"
            dot.node(or_id, label="OR", shape="ellipse", fillcolor="#FFCCCB")
            dot.edge(parent, or_id)
            dot.edge(or_id, children[0])
            dot.edge(or_id, children[1])
            edges_json.extend([
                {"from": parent, "to": or_id},
                {"from": or_id, "to": children[0]},
                {"from": or_id, "to": children[1]}
            ])
            node_counters['gate'] += 1
        else:
            mid = len(children) // 2
            left = children[:mid]
            right = children[mid:]
            or_id = f"or_{tactic_index}_{node_counters['gate']}"
            dot.node(or_id, label="OR", shape="ellipse", fillcolor="#FFCCCB")
            node_counters['gate'] += 1
            dot.edge(parent, or_id)
            edges_json.append({"from": parent, "to": or_id})
            binary_or(or_id, left, edges_json)
            binary_or(or_id, right, edges_json)

    cwe_ids = []
    for cwe in tactic_weaknesses:
        cwe_id = f"cwe_{tactic_index}_{node_counters['cwe']}"
        cwe_df_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
        metadata = extract_node_metadata(cwe_df_subset, "cwe")
        dot.node(cwe_id, label=cwe, fillcolor="#ADD8E6", shape="note")
        nodes_json.append({
            "id": cwe_id,
            "label": cwe,
            "type": "cwe",
            "metadata": metadata
        })
        cwe_ids.append(cwe_id)
        node_counters['cwe'] += 1

    binary_or(tactic_id, cwe_ids, edges_json)

    for i, cwe in enumerate(tactic_weaknesses):
        cwe_df = tactic_df[tactic_df["cwe_Name"] == cwe]
        techniques = cwe_df["technique_name"].dropna().unique().tolist()

        for technique in techniques:
            tech_df = cwe_df[cwe_df["technique_name"] == technique]
            tech_key = f"{tactic}:{technique}"
            if tech_key in added_techniques:
                continue
            added_techniques[tech_key] = True

            subtechniques = tech_df["subtechnique_name"].dropna().unique().tolist()
            subtechniques = [st for st in subtechniques if st.strip().lower() not in cwe_name_set]

            capec_names = tech_df["capec_Name"].dropna().unique().tolist()
            capec_name = capec_names[0] if len(capec_names) > 0 else "CAPEC ?"

            if subtechniques:
                for subtech in subtechniques:
                    subtech_id = f"subtech_{tactic_index}_{node_counters['subtech']}"
                    subtech_df_subset = tech_df[tech_df["subtechnique_name"] == subtech] if "subtechnique_name" in tech_df.columns else pd.DataFrame()
                    metadata = extract_node_metadata(subtech_df_subset, "subtech")
                    dot.node(subtech_id, label=subtech, fillcolor="#FFFACD", shape="box")
                    nodes_json.append({
                        "id": subtech_id,
                        "label": subtech,
                        "type": "subtech",
                        "metadata": metadata
                    })

                    capec_id = f"capec_{tactic_index}_{node_counters['capec']}"
                    capec_df_subset = tech_df[tech_df["capec_Name"] == capec_name] if "capec_Name" in tech_df.columns else pd.DataFrame()
                    metadata = extract_node_metadata(capec_df_subset, "capec")
                    dot.node(capec_id, label=capec_name, fillcolor="#FFB6C1", shape="box")
                    nodes_json.append({
                        "id": capec_id,
                        "label": capec_name,
                        "type": "capec",
                        "metadata": metadata
                    })

                    and_gate_id = f"and_{tactic_index}_{node_counters['gate']}"
                    dot.node(and_gate_id, label="AND", shape="ellipse", fillcolor="#FFCCCB")
                    dot.edge(cwe_ids[i], and_gate_id)
                    dot.edge(and_gate_id, subtech_id)
                    dot.edge(and_gate_id, capec_id)
                    edges_json.extend([
                        {"from": cwe_ids[i], "to": and_gate_id},
                        {"from": and_gate_id, "to": subtech_id},
                        {"from": and_gate_id, "to": capec_id}
                    ])
                    node_counters['subtech'] += 1
                    node_counters['capec'] += 1
                    node_counters['gate'] += 1
            else:
                tech_id = f"tech_{tactic_index}_{node_counters['tech']}"
                tech_df_subset = tactic_df[tactic_df["technique_name"] == technique]
                metadata = extract_node_metadata(tech_df_subset, "tech")
                dot.node(tech_id, label=technique, fillcolor="#90EE90", shape="box")
                nodes_json.append({
                    "id": tech_id,
                    "label": technique,
                    "type": "tech",
                    "metadata": metadata
                })

                capec_id = f"capec_{tactic_index}_{node_counters['capec']}"
                capec_df_subset = tech_df[tech_df["capec_Name"] == capec_name] if "capec_Name" in tech_df.columns else pd.DataFrame()
                metadata = extract_node_metadata(capec_df_subset, "capec")
                dot.node(capec_id, label=capec_name, fillcolor="#FFB6C1", shape="box")
                nodes_json.append({
                    "id": capec_id,
                    "label": capec_name,
                    "type": "capec",
                    "metadata": metadata
                })

                and_gate_id = f"and_{tactic_index}_{node_counters['gate']}"
                dot.node(and_gate_id, label="AND", shape="ellipse", fillcolor="#FFCCCB")
                dot.edge(cwe_ids[i], and_gate_id)
                dot.edge(and_gate_id, tech_id)
                dot.edge(and_gate_id, capec_id)
                edges_json.extend([
                    {"from": cwe_ids[i], "to": and_gate_id},
                    {"from": and_gate_id, "to": tech_id},
                    {"from": and_gate_id, "to": capec_id}
                ])
                node_counters['tech'] += 1
                node_counters['capec'] += 1
                node_counters['gate'] += 1

    return dot, added_techniques, (nodes_json, edges_json)

def replace_nan_with_string(obj):
    if isinstance(obj, dict):
        return {k: replace_nan_with_string(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan_with_string(elem) for elem in obj]
    elif pd.isna(obj):
        return "\"NaN\""
    else:
        return obj

# Generate graphs & JSON for all tactics
def generate_single_tactic_graphs():
    all_added_techniques = {}

    for i, tactic in enumerate(ordered_tactics):
        print(f"\nCreating graph for tactic {i + 1}: {tactic}")

        dot, added_techniques, (nodes_json, edges_json) = create_single_tactic_graph(tactic, i)

        all_added_techniques.update(added_techniques)

        filename_base = f"{output_dir}/attack_tree_tactic_{i+1:02d}_{tactic.replace(' ', '_')}"

        # Render PNG image
        dot.render(filename_base, format="png", cleanup=True)
        print(f"Saved graph image: {filename_base}.png")

        # Replace NaNs with string and save JSON
        clean_nodes_json = replace_nan_with_string(nodes_json)
        clean_edges_json = replace_nan_with_string(edges_json)

        json_data = {
            "tactic": tactic,
            "nodes": clean_nodes_json,
            "edges": clean_edges_json,
        }

        json_path = f"{filename_base}.json"
        with open(json_path, "w") as f:
            json.dump(json_data, f, indent=4)
        print(f"Saved graph metadata JSON: {json_path}")

    print("\nAdded techniques summary:")
    for tech in all_added_techniques.keys():
        print(f"- {tech}")

# Run the generator
generate_single_tactic_graphs()


Processing 8 tactics in this order: ['Initial Access', 'Persistence', 'Privilege Escalation', 'Defense Evasion', 'Credential Access', 'Discovery', 'Lateral Movement', 'Collection']

Creating graph for tactic 1: Initial Access
Saved graph image: attack_tree_images/attack_tree_tactic_01_Initial_Access.png
Saved graph metadata JSON: attack_tree_images/attack_tree_tactic_01_Initial_Access.json

Creating graph for tactic 2: Persistence
Saved graph image: attack_tree_images/attack_tree_tactic_02_Persistence.png
Saved graph metadata JSON: attack_tree_images/attack_tree_tactic_02_Persistence.json

Creating graph for tactic 3: Privilege Escalation
Saved graph image: attack_tree_images/attack_tree_tactic_03_Privilege_Escalation.png
Saved graph metadata JSON: attack_tree_images/attack_tree_tactic_03_Privilege_Escalation.json

Creating graph for tactic 4: Defense Evasion
Saved graph image: attack_tree_images/attack_tree_tactic_04_Defense_Evasion.png
Saved graph metadata JSON: attack_tree_images/at

# NEO4J Two-Tactic Attack Tree Creator

In [19]:
# import pandas as pd
# import re
# import hashlib
# from neo4j import GraphDatabase

# # -------------------- CONFIG --------------------
# NEO4J_URI = "neo4j://127.0.0.1:7687"
# NEO4J_USER = "neo4j"
# NEO4J_PASSWORD = "Swapnil@305"   # Change to your password
# DATASET_PATH = "device_compromise_dataset.xlsx"

In [20]:
# mitre_tactic_order = [
#     "Reconnaissance", "Resource Development", "Initial Access", "Execution", "Persistence", 
#     "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery", 
#     "Lateral Movement", "Collection", "Command and Control", "Exfiltration", "Impact"
# ]

In [21]:


# # Load dataset
# df = pd.read_excel(DATASET_PATH, index_col=False)
# df["tactics"] = df["tactics"].str.strip()

# # Tactics to process
# TACTICS = ["Defense Evasion", "Initial Access"]

# # ==== HELPER FUNCTIONS ====
# def clean_id(text):
#     """Clean IDs to avoid spaces/special chars that can cause mismatches."""
#     if pd.isna(text):
#         return "NA"
#     return re.sub(r"[^A-Za-z0-9_\-]", "_", str(text))

# def extract_node_metadata(df_subset, node_type):
#     def get_col(col):
#         if col in df_subset.columns and not df_subset.empty:
#             val = df_subset[col].iloc[0]
#             return None if pd.isna(val) else val
#         return None

#     keys_map = {
#         "cwe": [
#             'CVE ID','Description','cwe_Name','cwe_Abstraction','cwe_Structure','cwe_Status',
#             'cwe_Description','cwe_Extended_Description','cwe_Related_Weaknesses','cwe_Weakness_Ordinality',
#             'cwe_Language_Name','cwe_Technology_Class','cwe_Alternate_Terms','cwe_Alternate_Terms_With_Descriptions',
#             'cwe_Background_Details','cwe_Modes_Of_Introduction','cwe_Likelihood_Of_Exploit','cwe_Common_Consequences',
#             'cwe_Detection_Methods','cwe_Potential_Mitigations','cwe_Demonstrative_Examples','cwe_Observed_Examples',
#             'attackVector','baseScore','baseSeverity','attackComplexity','privilegesRequired',
#             'userInteraction','confidentialityImpact','availabilityImpact','exploitabilityScore'
#         ],
#         "capec": [
#             'CVE ID','capec_Name','capec_Mitre_Techniques','capec_Abstraction','capec_Status',
#             'capec_Description','capec_Extended_Description','capec_Likelihood_Of_Attack','capec_Typical_Severity',
#             'capec_Execution_Flow','capec_Prerequisites','capec_Skills_Required','capec_Resources_Required',
#             'capec_Consequences','capec_Mitigations','capec_Example_Instances','capec_Related_Weaknesses',
#             'capec_Taxonomy_Mappings'
#         ],
#         "tech": [
#             'CVE ID','Description','technique_name','subtechnique_name','tactics','detection',
#             'platforms','data sources','defenses bypassed','contributors','permissions required',
#             'supports remote','system requirements','impact type','effective permissions',
#             'capec_Mitre_Techniques','attackVector','baseScore','baseSeverity','attackComplexity',
#             'privilegesRequired','userInteraction','confidentialityImpact','availabilityImpact','exploitabilityScore'
#         ],
#         "subtech": [
#             'subtechnique_name','technique_name','tactics','detection','platforms','data sources'
#         ]
#     }
#     keys = keys_map.get(node_type, [])
#     return {k: str(get_col(k)) for k in keys if get_col(k) is not None}

# # ==== NEO4J SETUP ====
# driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# def create_node(tx, label, node_id, props):
#     print(f"[CREATE NODE] {label} ({node_id})")
#     tx.run(f"MERGE (n:{label} {{id:$id}}) SET n += $props", id=node_id, props=props)

# def create_relationship(tx, from_id, to_id):
#     # Verify target exists
#     res1 = tx.run("MATCH (a {id:$id}) RETURN a", id=from_id).single()
#     res2 = tx.run("MATCH (b {id:$id}) RETURN b", id=to_id).single()
#     assert res1 is not None, f"Parent node {from_id} does not exist!"
#     assert res2 is not None, f"Child node {to_id} does not exist!"
#     print(f"[REL] {from_id} -> {to_id}")
#     tx.run("""
#         MATCH (a {id:$from_id}), (b {id:$to_id})
#         MERGE (a)-[:HAS_CHILD]->(b)
#     """, from_id=from_id, to_id=to_id)

# def create_gate(tx, gate_id, gate_type):
#     print(f"[CREATE GATE] {gate_type} ({gate_id})")
#     tx.run("MERGE (g:Gate {id:$id}) SET g.type=$type, g.label=$type", id=gate_id, type=gate_type)

# def build_or_chain(ids, prefix):
#     if not ids:
#         return None, []
#     if len(ids) == 1:
#         return ids[0], []
#     if len(ids) == 2:
#         return f"{prefix}_or_gate", [("OR", f"{prefix}_or_gate", ids)]
#     mid = len(ids) // 2
#     left_id, left_ops = build_or_chain(ids[:mid], prefix + "_L")
#     right_id, right_ops = build_or_chain(ids[mid:], prefix + "_R")
#     gate_id = f"{prefix}_or_gate"
#     return gate_id, [("OR", gate_id, [left_id, right_id])] + left_ops + right_ops

# # ==== MAIN BUILD FUNCTION ====
# def push_attack_tree():
#     with driver.session() as session:
#         def transaction(tx):
#             # Root and SAND
#             root_id = "root"
#             create_node(tx, "Root", root_id, {"id": root_id, "label": "Compromise Device", "type": "root"})
#             sand_gate_id = "sand_gate_root"
#             create_gate(tx, sand_gate_id, "SAND")
#             create_relationship(tx, root_id, sand_gate_id)

#             tactic_ids = []
#             for idx, tactic in enumerate(TACTICS):
#                 tactic_id = f"tactic_{idx}"
#                 create_node(tx, "Tactic", tactic_id, {"id": tactic_id, "label": tactic, "type": "tactic"})
#                 create_relationship(tx, sand_gate_id, tactic_id)
#                 tactic_ids.append(tactic_id)

#             # Process each tactic
#             for idx, tactic in enumerate(TACTICS):
#                 print(f"\n=== Processing Tactic: {tactic} ===")
#                 tactic_id = tactic_ids[idx]
#                 tactic_df = df[df["tactics"] == tactic]
#                 cwe_names = tactic_df["cwe_Name"].dropna().unique()
#                 cwe_ids = []

#                 # Create CWEs
#                 for c_idx, cwe in enumerate(cwe_names):
#                     cwe_id = f"{tactic_id}_cwe_{c_idx}_{clean_id(cwe)}"
#                     cwe_df = tactic_df[tactic_df["cwe_Name"] == cwe]
#                     create_node(tx, "CWE", cwe_id, {"id": cwe_id, "label": cwe, "type": "cwe", **extract_node_metadata(cwe_df, "cwe")})
#                     cwe_ids.append(cwe_id)

#                 # Connect CWEs with OR gates created first, then relationships
#                 if len(cwe_ids) == 1:
#                     create_relationship(tx, tactic_id, cwe_ids[0])
#                 elif len(cwe_ids) > 1:
#                     or_root_id, or_ops = build_or_chain(cwe_ids, f"{tactic_id}_cwe")

#                     # First create all OR gate nodes
#                     for gate_type, gate_id, children in or_ops:
#                         create_gate(tx, gate_id, gate_type)

#                     # Then create all relationships parent->child between gates and CWEs
#                     for gate_type, gate_id, children in or_ops:
#                         for child_id in children:
#                             create_relationship(tx, gate_id, child_id)

#                     create_relationship(tx, tactic_id, or_root_id)

#                 cwe_name_set = set(c.lower() for c in cwe_names)

#                 # CWE children
#                 for c_idx, cwe in enumerate(cwe_names):
#                     cwe_id = cwe_ids[c_idx]
#                     cwe_df = tactic_df[tactic_df["cwe_Name"] == cwe]
#                     techniques = cwe_df["technique_name"].dropna().unique()

#                     for tech in techniques:
#                         tech_id_clean = clean_id(tech)
#                         tech_df = cwe_df[cwe_df["technique_name"] == tech]
#                         subtechniques = []

#                         if "subtechnique_name" in tech_df.columns:
#                             subtech_raw = tech_df["subtechnique_name"].dropna().unique()
#                             subtechniques = [st for st in subtech_raw if st.strip().lower() not in cwe_name_set]

#                         capec_names = tech_df["capec_Name"].dropna().unique()
#                         capec_name = capec_names[0] if len(capec_names) > 0 else "CAPEC_Unknown"
#                         capec_id_clean = clean_id(capec_name)

#                         if subtechniques:
#                             for subtech in subtechniques:
#                                 subtech_id = f"{cwe_id}_subtech_{clean_id(subtech)}"
#                                 subtech_df = tech_df[tech_df["subtechnique_name"] == subtech]
#                                 create_node(tx, "Subtech", subtech_id, {"id": subtech_id, "label": subtech,
#                                                                          "type": "subtech", **extract_node_metadata(subtech_df, "subtech")})
#                                 capec_id = f"{cwe_id}_capec_{capec_id_clean}"
#                                 capec_df = tech_df[tech_df["capec_Name"] == capec_name]
#                                 create_node(tx, "CAPEC", capec_id, {"id": capec_id, "label": capec_name,
#                                                                     "type": "capec", **extract_node_metadata(capec_df, "capec")})
#                                 and_gate_id = f"{cwe_id}_and_{clean_id(subtech)}"
#                                 create_gate(tx, and_gate_id, "AND")
#                                 create_relationship(tx, cwe_id, and_gate_id)
#                                 create_relationship(tx, and_gate_id, subtech_id)
#                                 create_relationship(tx, and_gate_id, capec_id)
#                         else:
#                             tech_node_id = f"{cwe_id}_tech_{tech_id_clean}"
#                             create_node(tx, "Tech", tech_node_id, {"id": tech_node_id, "label": tech,
#                                                                    "type": "tech", **extract_node_metadata(tech_df, "tech")})
#                             capec_id = f"{cwe_id}_capec_{capec_id_clean}"
#                             capec_df = tech_df[tech_df["capec_Name"] == capec_name]
#                             create_node(tx, "CAPEC", capec_id, {"id": capec_id, "label": capec_name,
#                                                                 "type": "capec", **extract_node_metadata(capec_df, "capec")})
#                             and_gate_id = f"{cwe_id}_and_{tech_id_clean}"
#                             create_gate(tx, and_gate_id, "AND")
#                             create_relationship(tx, cwe_id, and_gate_id)
#                             create_relationship(tx, and_gate_id, tech_node_id)
#                             create_relationship(tx, and_gate_id, capec_id)

#             # Post-creation integrity check
#             print("\n=== POST-BUILD CONNECTIVITY CHECK ===")
#             unreachable = tx.run("""
#                 MATCH (n) 
#                 WHERE NOT (n)<-[:HAS_CHILD*]-(:Root) AND NOT n:Root
#                 RETURN n.id AS node_id, labels(n) AS labels
#             """).data()
#             if unreachable:
#                 print("[WARNING] Found unreachable nodes:")
#                 for rec in unreachable:
#                     print(rec)
#             else:
#                 print("All nodes are connected to Root.")

#         session.execute_write(transaction)

# if __name__ == "__main__":
#     push_attack_tree()


# HTML INTERACTIVE TREE

In [22]:
#!/usr/bin/env python3
"""
comprehensive_attack_tree_builder.py

Generates:
 - comprehensive_attack_tree.svg  (Graphviz static layout w/ hover showing label only)
 - comprehensive_attack_tree.html (embeds the SVG; click nodes to see metadata in a right-side panel)
 - comprehensive_attack_tree.png  (optional, rasterized via Graphviz)

Place `device_compromise_dataset.xlsx` in the same folder or update DATASET_PATH.
"""
import os
import re
import json
import html as html_lib
import pandas as pd
from graphviz import Digraph
import xml.etree.ElementTree as ET

# -------------------------
# Config / file paths
# -------------------------
DATASET_PATH = "device_compromise_dataset.xlsx"
OUTPUT_DIR = "attack_tree_html_unified"
SVG_BASENAME = "comprehensive_attack_tree"
SVG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".svg")
HTML_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".html")
PNG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".png")

# MITRE tactic ordering (keeps consistent ordering)
MITRE_TACTIC_ORDER = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution", "Persistence",
    "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery",
    "Lateral Movement", "Collection", "Command and Control", "Exfiltration", "Impact"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# Read dataset
# -------------------------
df = pd.read_excel(DATASET_PATH, index_col=False)
df = df.fillna("")  # replace NaN with empty strings for simpler checks

if "tactics" not in df.columns:
    raise KeyError("Expected 'tactics' column in the dataset (case-sensitive).")

df["tactics"] = df["tactics"].astype(str).str.strip()

# Determine ordered tactics present in dataset
unique_tactics = [t for t in df["tactics"].unique() if t != ""]
ordered_tactics = [t for t in MITRE_TACTIC_ORDER if t in unique_tactics]
for t in unique_tactics:
    if t not in ordered_tactics:
        ordered_tactics.append(t)

print("Tactics (ordered):", ordered_tactics)

# -------------------------
# Helper functions
# -------------------------
def _normalize_name(s: str) -> str:
    """Normalize a column/key name: lowercase alphanum only to match flexible variants."""
    return re.sub(r'[^0-9a-z]', '', str(s).lower())

def extract_node_metadata(df_subset: pd.DataFrame, node_type: str):
    """
    Extract metadata for a node subset.

    Aggregate unique non-empty values across all rows for each key.
    Column matching is tolerant to case/spacing/underscore variations.
    """
    def build_cols_map(cols):
        return { _normalize_name(c): c for c in cols }

    cols_map = build_cols_map(df_subset.columns.tolist())

    if node_type == "cwe":
        keys = [
            'CVE ID', 'Description', 'cwe_Name', 'cwe_Abstraction', 'cwe_Structure', 'cwe_Status',
            'cwe_Description', 'cwe_Extended_Description', 'cwe_Related_Weaknesses', 'cwe_Weakness_Ordinality',
            'cwe_Language_Name', 'cwe_Technology_Class', 'cwe_Alternate_Terms', 'cwe_Alternate_Terms_With_Descriptions',
            'cwe_Background_Details', 'cwe_Modes_Of_Introduction', 'cwe_Likelihood_Of_Exploit',
            'cwe_Common_Consequences', 'cwe_Detection_Methods', 'cwe_Potential_Mitigations', 'cwe_Demonstrative_Examples',
            'cwe_Observed_Examples', 'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity',
            'privilegesRequired', 'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "capec":
        keys = [
            'CVE ID', 'capec_Name', 'capec_Mitre_Techniques', 'capec_Abstraction', 'capec_Status',
            'capec_Description', 'capec_Extended_Description', 'capec_Likelihood_Of_Attack', 'capec_Typical_Severity',
            'capec_Execution_Flow', 'capec_Prerequisites', 'capec_Skills_Required', 'capec_Resources_Required',
            'capec_Consequences', 'capec_Mitigations', 'capec_Example_Instances', 'capec_Related_Weaknesses',
            'capec_Taxonomy_Mappings'
        ]
    elif node_type == "tech":
        keys = [
            'CVE ID', 'Description', 'tech_description','technique_name', 'subtechnique_name', 'tactics', 'detection',
            'platforms', 'data sources', 'defenses bypassed', 'contributors', 'permissions required',
            'supports remote', 'system requirements', 'impact type', 'effective permissions',
            'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity', 'privilegesRequired',
            'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "subtech":
        keys = [
            'subtechnique_name', 'technique_name', 'tactics', 'detection', 'platforms', 'data sources'
        ]
    else:
        keys = df_subset.columns.tolist()

    meta = {}
    for k in keys:
        nk = _normalize_name(k)
        if nk in cols_map:
            colname = cols_map[nk]
        else:
            colname = None
        if colname and colname in df_subset.columns:
            vals = df_subset[colname].astype(str).map(str.strip)
            vals = [v for v in vals if v != "" and v.lower() != "nan"]
            uniq = []
            for v in vals:
                if v not in uniq:
                    uniq.append(v)
            if len(uniq) == 1:
                meta[k] = uniq[0]
            elif len(uniq) > 1:
                meta[k] = " | ".join(uniq)
    return meta

def format_metadata_for_html(metadata_dict, max_rows=60):
    """
    Convert metadata dict to an HTML table (string). Escapes values for safety.
    """
    if not metadata_dict:
        return "<div class='meta-empty'>(No metadata available)</div>"
    rows = []
    preferred_order = ["cwe_Name", "capec_Name", "technique_name", "subtechnique_name", "Description", "CVE ID"]
    added = set()
    for k in preferred_order:
        if k in metadata_dict:
            rows.append((k, metadata_dict[k])); added.add(k)
    for k, v in metadata_dict.items():
        if k in added: continue
        if len(rows) >= max_rows: break
        rows.append((k, v))
    table_html = ["<table class='meta-table' style='width:100%;border-collapse:collapse;'>"]
    for k, v in rows:
        k_esc = html_lib.escape(str(k))
        v_esc = html_lib.escape(str(v))
        table_html.append(
            f"<tr><th style='text-align:left;padding:8px;border-bottom:1px solid #eee;width:38%;vertical-align:top;background:#fafafa'>{k_esc}</th>"
            f"<td style='padding:8px;border-bottom:1px solid #eee;vertical-align:top'>{v_esc}</td></tr>"
        )
    table_html.append("</table>")
    return "\n".join(table_html)

def build_gate_tree(dot, children, gate_label, gate_prefix, gate_counter, nodes_data, edges_data, gate_color="#FFCCCB"):
    """
    Build a minimal binary tree of gate nodes representing a choice among `children`.
    Returns the root node id representing the OR (or the single child's id if only one child).
    Guarantees exactly one top-level gate for the group (no accidental double-wrapping).
    """
    if not children:
        return None
    if len(children) == 1:
        return children[0]
    def _build(sub_children):
        if len(sub_children) == 1:
            return sub_children[0]
        g_id = f"{gate_prefix}_{gate_counter[0]}"
        gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        mid = len(sub_children) // 2
        left = _build(sub_children[:mid])
        right = _build(sub_children[mid:])
        if left:
            dot.edge(g_id, left); edges_data.append({"from": g_id, "to": left})
        if right:
            dot.edge(g_id, right); edges_data.append({"from": g_id, "to": right})
        return g_id
    return _build(children)

def binary_gate(dot, parent, children, gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix):
    """
    Backwards-compatible parent->gate->children connector used in several places.
    """
    if len(children) == 0:
        return
    if len(children) == 1:
        dot.edge(parent, children[0]); edges_data.append({"from": parent, "to": children[0]}); return
    if len(children) == 2:
        g_id = f"{gate_prefix}_{gate_counter[0]}"; gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        dot.edge(parent, g_id); dot.edge(g_id, children[0]); dot.edge(g_id, children[1])
        edges_data.extend([{"from": parent, "to": g_id}, {"from": g_id, "to": children[0]}, {"from": g_id, "to": children[1]}])
        return
    mid = len(children) // 2
    g_id = f"{gate_prefix}_{gate_counter[0]}"; gate_counter[0] += 1
    dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
    nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
    dot.edge(parent, g_id); edges_data.append({"from": parent, "to": g_id})
    binary_gate(dot, g_id, children[:mid], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)
    binary_gate(dot, g_id, children[mid:], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)

# -------------------------
# Build combined graph
# -------------------------
def build_attack_tree():
    dot = Digraph(name="comprehensive_attack_tree", format="svg")
    dot.attr('graph', rankdir='TB', splines='ortho', nodesep='0.6', ranksep='1.0')
    dot.attr('node', fontname='Helvetica', fontsize='28')
    dot.attr('edge', fontsize='8')

    nodes_data = []
    edges_data = []
    node_metadata_map = {}

    # Root node
    root_id = "root"
    dot.node(root_id, label="Compromise Device", shape="box", fillcolor="#FFD700", style="filled")
    nodes_data.append({"id": root_id, "label": "Compromise Device", "type": "root", "metadata": {"Description": "Root node"}})

    # Tactic nodes
    tactic_ids = []
    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        dot.node(tid, label=tactic, shape="box", fillcolor="#98FB98", style="filled")
        nodes_data.append({"id": tid, "label": tactic, "type": "tactic", "metadata": {"tactic": tactic}})
        tactic_ids.append(tid)

    # Connect root -> tactics using binary SAND gates
    sand_counter = [0]
    binary_gate(dot, root_id, tactic_ids, "SAND", "#FFCCCB", sand_counter, nodes_data, edges_data, gate_prefix="sand")

    or_counter = [0]
    and_counter = [0]

    # For each tactic, attach CWEs and under each CWE attach technique/subtech + CAPEC
    for ti, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{ti}"
        tactic_df = df[df["tactics"] == tactic]

        # list of unique cwe names (defensive: use .get to avoid keyerrors)
        cwe_names = [x for x in tactic_df.get("cwe_Name", pd.Series([], dtype=object)).unique().tolist() if x != ""]
        cwe_ids = []
        for ci, cwe in enumerate(cwe_names):
            cwe_id = f"cwe_{ti}_{ci}"
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
            meta = extract_node_metadata(cwe_subset, "cwe")
            dot.node(cwe_id, label=cwe, shape="note", fillcolor="#ADD8E6", style="filled")
            nodes_data.append({"id": cwe_id, "label": cwe, "type": "cwe", "metadata": meta})
            node_metadata_map[cwe_id] = format_metadata_for_html(meta)
            cwe_ids.append(cwe_id)

        # Connect tactic -> cwe(s) using an OR gate (binary style)
        if cwe_ids:
            binary_gate(dot, tid, cwe_ids, "OR", "#FFCCCB", or_counter, nodes_data, edges_data, gate_prefix="or")

        # For each CWE, process techniques
        for ci, cwe in enumerate(cwe_names):
            cwe_id = f"cwe_{ti}_{ci}"
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]

            techniques = [x for x in cwe_subset.get("technique_name", pd.Series([], dtype=object)).unique().tolist() if x != ""]
            for t_idx, tech in enumerate(techniques):
                tech_subset = cwe_subset[cwe_subset["technique_name"] == tech]

                # subtechniques present?
                subtechs = [s for s in tech_subset.get("subtechnique_name", pd.Series([], dtype=object)).unique().tolist() if s != ""]

                if subtechs:
                    for s_i, subtech in enumerate(subtechs):
                        sub_id = f"subtech_{ti}_{ci}_{t_idx}_{s_i}"
                        sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech]
                        sub_meta = extract_node_metadata(sub_rows, "subtech")
                        dot.node(sub_id, label=subtech, shape="box", fillcolor="#FFFACD", style="filled")
                        nodes_data.append({"id": sub_id, "label": subtech, "type": "subtech", "metadata": sub_meta})
                        node_metadata_map[sub_id] = format_metadata_for_html(sub_meta)

                        capec_names = [c for c in sub_rows.get("capec_Name", pd.Series([], dtype=object)).unique().tolist() if c != ""]
                        capec_nodes = []
                        for cap_i, cap in enumerate(capec_names):
                            cap_id = f"capec_{ti}_{ci}_{t_idx}_{s_i}_{cap_i}"
                            cap_rows = sub_rows[sub_rows["capec_Name"] == cap]
                            cap_meta = extract_node_metadata(cap_rows, "capec")
                            dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                            nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                            node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                            capec_nodes.append(cap_id)

                        if not capec_nodes:
                            cap_id = f"capec_{ti}_{ci}_{t_idx}_{s_i}_na"
                            dot.node(cap_id, label="CAPEC ?", shape="box", fillcolor="#FFB6C1", style="filled")
                            nodes_data.append({"id": cap_id, "label": "CAPEC ?", "type": "capec", "metadata": {}})
                            node_metadata_map[cap_id] = "<div class='meta-empty'>(No metadata)</div>"
                            capec_nodes = [cap_id]

                        # build a single OR-tree representing choice among capec nodes (root id returned)
                        cap_rep_node = build_gate_tree(dot, capec_nodes, "OR", "orcap", or_counter, nodes_data, edges_data)

                        # AND gate: cwe -> (subtech & cap_rep_node)
                        and_id = f"and_{ti}_{ci}_{t_idx}_{s_i}_{and_counter[0]}"
                        and_counter[0] += 1
                        dot.node(and_id, label="AND", shape="ellipse", fillcolor="#FFCCCB", style="filled")
                        nodes_data.append({"id": and_id, "label": "AND", "type": "gate"})
                        dot.edge(cwe_id, and_id); edges_data.append({"from": cwe_id, "to": and_id})
                        dot.edge(and_id, sub_id); edges_data.append({"from": and_id, "to": sub_id})
                        if cap_rep_node:
                            dot.edge(and_id, cap_rep_node); edges_data.append({"from": and_id, "to": cap_rep_node})
                else:
                    # No subtechs: use technique node
                    tech_id = f"tech_{ti}_{ci}_{t_idx}"
                    tech_meta = extract_node_metadata(tech_subset, "tech")
                    dot.node(tech_id, label=tech, shape="box", fillcolor="#90EE90", style="filled")
                    nodes_data.append({"id": tech_id, "label": tech, "type": "tech", "metadata": tech_meta})
                    node_metadata_map[tech_id] = format_metadata_for_html(tech_meta)

                    capec_names = [c for c in tech_subset.get("capec_Name", pd.Series([], dtype=object)).unique().tolist() if c != ""]
                    capec_nodes = []
                    for cap_i, cap in enumerate(capec_names):
                        cap_id = f"capec_{ti}_{ci}_{t_idx}_tcap_{cap_i}"
                        cap_rows = tech_subset[tech_subset["capec_Name"] == cap]
                        cap_meta = extract_node_metadata(cap_rows, "capec")
                        dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                        nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                        node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                        capec_nodes.append(cap_id)

                    if not capec_nodes:
                        cap_id = f"capec_{ti}_{ci}_{t_idx}_tcap_na"
                        dot.node(cap_id, label="CAPEC ?", shape="box", fillcolor="#FFB6C1", style="filled")
                        nodes_data.append({"id": cap_id, "label": "CAPEC ?", "type": "capec", "metadata": {}})
                        node_metadata_map[cap_id] = "<div class='meta-empty'>(No metadata)</div>"
                        capec_nodes = [cap_id]

                    # build a single OR-tree representing choice among capec nodes (root id returned)
                    cap_rep_node = build_gate_tree(dot, capec_nodes, "OR", "orcap", or_counter, nodes_data, edges_data)

                    # AND gate: cwe -> (tech & cap_rep_node)
                    and_id = f"and_{ti}_{ci}_{t_idx}_tech_{and_counter[0]}"
                    and_counter[0] += 1
                    dot.node(and_id, label="AND", shape="ellipse", fillcolor="#FFCCCB", style="filled")
                    nodes_data.append({"id": and_id, "label": "AND", "type": "gate"})
                    dot.edge(cwe_id, and_id); edges_data.append({"from": cwe_id, "to": and_id})
                    dot.edge(and_id, tech_id); edges_data.append({"from": and_id, "to": tech_id})
                    if cap_rep_node:
                        dot.edge(and_id, cap_rep_node); edges_data.append({"from": and_id, "to": cap_rep_node})

    # -------------------------
    # Post-build pass: compute tactic reachability (CWEs, techniques, CAPECs)
    # -------------------------
    adj = {}
    for e in edges_data:
        u = e.get("from"); v = e.get("to")
        if u is None or v is None: continue
        adj.setdefault(u, []).append(v)

    id_to_type = { nd["id"]: nd.get("type", "") for nd in nodes_data }
    id_to_label = { nd["id"]: nd.get("label", nd["id"]) for nd in nodes_data }

    def reachable_nodes(start_id):
        seen = set()
        q = [start_id]
        seen.add(start_id)
        result = []
        while q:
            cur = q.pop(0)
            for nb in adj.get(cur, []):
                if nb not in seen:
                    seen.add(nb)
                    result.append(nb)
                    q.append(nb)
        return result

    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        reached = reachable_nodes(tid)
        cwe_labels = []
        tech_labels = []
        capec_labels = []
        for nid in reached:
            ntype = id_to_type.get(nid, "")
            nlabel = id_to_label.get(nid, nid)
            if ntype == "cwe":
                if nlabel not in cwe_labels: cwe_labels.append(nlabel)
            elif ntype in ("tech", "subtech"):
                if nlabel not in tech_labels: tech_labels.append(nlabel)
            elif ntype == "capec":
                if nlabel not in capec_labels: capec_labels.append(nlabel)
        tactic_meta = {
            "tactic": tactic,
            "cwes": ", ".join(cwe_labels) if cwe_labels else "",
            "techniques": ", ".join(tech_labels) if tech_labels else "",
            "capecs": ", ".join(capec_labels) if capec_labels else ""
        }
        for nd in nodes_data:
            if nd["id"] == tid:
                nd_meta = nd.get("metadata", {}).copy() if isinstance(nd.get("metadata", {}), dict) else {}
                nd_meta.update(tactic_meta)
                nd["metadata"] = nd_meta
                break
        node_metadata_map[tid] = format_metadata_for_html(tactic_meta)
    # -------------------------

    return dot, nodes_data, edges_data, node_metadata_map

# -------------------------
# SVG post-processing: inject attributes (data-node-id), set harmless title and collect positions
# -------------------------
def inject_tooltips_and_extract_positions(svg_path, node_metadata_map):
    ns = {'svg': 'http://www.w3.org/2000/svg'}
    ET.register_namespace("", ns['svg'])
    tree = ET.parse(svg_path)
    root = tree.getroot()

    node_positions = {}

    for g in root.findall(".//svg:g", ns):
        title = g.find("svg:title", ns)
        if title is None:
            continue
        original_title_text = title.text if title.text is not None else ""
        node_id = original_title_text

        # Find text label inside group (fallback)
        tt = g.find(".//svg:text", ns)
        label_text = tt.text if tt is not None and tt.text is not None else node_id

        # Set <title> to label only (keeps hover small)
        title.text = label_text

        # Tag group with data-node-id attribute
        if node_id:
            g.set("data-node-id", node_id)

        # extract translate(x,y) coordinates when present
        transform = g.get("transform", "")
        m = re.search(r"translate\(\s*([0-9\.\-]+)[,\s]+([0-9\.\-]+)\s*\)", transform)
        if m:
            try:
                x = float(m.group(1)); y = float(m.group(2))
                node_positions[node_id] = {"x": x, "y": y}
            except Exception:
                pass

    tree.write(svg_path, encoding="utf-8", xml_declaration=True)
    return node_positions

# -------------------------
# HTML wrapper creation (larger display) with right-side panel metadata JS
# -------------------------
def save_html_with_svg(svg_path, html_path, node_positions, nodes_data, node_metadata_map):
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()

    # node_metadata_map values already contain HTML (created by format_metadata_for_html)
    meta_json = json.dumps(node_metadata_map)
    positions_json = json.dumps(node_positions)
    nodes_data_json = json.dumps(nodes_data)

    # We now create a two-column flex layout: .main (left, expanding) and #panel-root (right, fixed width)
    script_blob = f"""
<script>
window.ATTACK_TREE_META = {{
  "node_positions": {positions_json},
  "short_metadata": {meta_json},
  "nodes_data": {nodes_data_json}
}};

document.addEventListener('DOMContentLoaded', function() {{
  // find panel root (placeholder)
  let panelRoot = document.getElementById('panel-root');
  if (!panelRoot) {{
    // fallback: create one at end of body
    panelRoot = document.createElement('div');
    panelRoot.id = 'panel-root';
    document.body.appendChild(panelRoot);
  }}

  // create panel element inside panelRoot (so layout is two-column)
  let panel = document.createElement('div');
  panel.id = 'metadata-panel';
  panel.style.boxSizing = 'border-box';
  panel.style.height = '100%';
  panel.style.width = '100%';
  panel.style.background = '#fff';
  panel.style.overflow = 'auto';
  panel.style.padding = '14px';
  panel.style.fontFamily = 'Arial, Helvetica, sans-serif';
  panel.style.fontSize = '15px';
  panel.style.display = 'none';
  panelRoot.appendChild(panel);

  // header with close button
  let header = document.createElement('div');
  header.style.display = 'flex';
  header.style.alignItems = 'center';
  header.style.justifyContent = 'space-between';
  header.style.marginBottom = '10px';

  let title = document.createElement('div');
  title.innerText = 'Node metadata';
  title.style.fontWeight = '700';
  title.style.fontSize = '18px';
  header.appendChild(title);

  let closeBtn = document.createElement('button');
  closeBtn.innerHTML = '&times;';
  closeBtn.title = 'Close';
  closeBtn.style.border = 'none';
  closeBtn.style.background = 'transparent';
  closeBtn.style.fontSize = '24px';
  closeBtn.style.cursor = 'pointer';
  closeBtn.style.lineHeight = '1';
  closeBtn.style.padding = '0 6px';
  closeBtn.addEventListener('click', function() {{ panel.style.display = 'none'; }});
  header.appendChild(closeBtn);

  panel.appendChild(header);

  let content = document.createElement('div');
  content.id = 'metadata-panel-content';
  panel.appendChild(content);

  let foot = document.createElement('div');
  foot.style.marginTop = '12px';
  foot.style.fontSize = '13px';
  foot.style.color = '#666';
  foot.innerText = 'Click a node in the graph to view metadata here. Press × to close.';
  panel.appendChild(foot);

  // Attach click listeners to any svg group that has data-node-id
  let svgWrap = document.querySelector('.svg-wrap');
  if (!svgWrap) return;
  let nodeGroups = svgWrap.querySelectorAll('[data-node-id]');
  nodeGroups.forEach(function(g) {{
    g.style.cursor = 'pointer';
    g.addEventListener('click', function(ev) {{
      ev.stopPropagation();
      let nodeId = g.getAttribute('data-node-id');
      let html = window.ATTACK_TREE_META.short_metadata[nodeId] || "";
      if (!html || html === "") {{
        let found = (window.ATTACK_TREE_META.nodes_data || []).find(x => x.id === nodeId);
        if (found && found.label) {{
          html = '<div style="font-weight:600;margin-bottom:8px">' + found.label + '</div>';
        }} else {{
          html = '<div class="meta-empty">(No metadata)</div>';
        }}
      }}
      let subtitle = '<div style="color:#555;font-size:13px;margin-bottom:10px">ID: ' + nodeId + '</div>';
      content.innerHTML = subtitle + html;
      panel.style.display = 'block';
      panel.scrollTop = 0;
    }});
  }});

  // Optional: close panel with Escape
  document.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') panel.style.display = 'none';
  }});
}});
</script>
"""

    # CSS & layout: use flex to let svg-wrap expand and occupy remaining screen space
    html_doc = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Comprehensive Attack Tree</title>
  <style>
    html, body {{ height: 100%; margin: 0; padding: 0; }}
    body {{ font-family: Arial, Helvetica, sans-serif; background: #fafafa; font-size:16px; }}
    .app {{ display:flex; height: calc(100vh - 20px); gap: 12px; padding: 10px; box-sizing: border-box; }}
    .main {{ flex: 1 1 auto; display:flex; flex-direction: column; min-width: 200px; }}
    .topbar {{ padding-bottom: 8px; }}
    h1 {{ font-size: 24px; margin: 0 0 6px 0; }}
    .legend {{ margin-top: 6px; font-size: 15px; }}
    .legend span {{ display: inline-block; margin-right: 12px; padding: 4px 8px; border-radius:4px; font-size:14px; }}
    p.note {{ font-size: 15px; color: #333; margin-top:8px; }}
    /* svg-wrap expands to fill available vertical space */
    .svg-wrap {{ flex: 1 1 auto; border: 1px solid #ddd; padding: 6px; background: #fff; overflow: auto; min-height: 200px; }}
    /* Let the inline SVG scale to the container height */
    .svg-wrap svg {{ height: 100%; width: auto; display: block; }}
    /* Panel root (right column) is fixed width but responsive on small screens */
    #panel-root {{ flex: 0 0 420px; max-width: 42vw; min-width: 320px; display:flex; flex-direction:column; box-sizing: border-box; }}
    @media (max-width: 1000px) {{
      #panel-root {{ min-width: 260px; max-width: 40vw; }}
      .legend span {{ font-size:13px; }}
    }}
    /* small metadata table styles (panel uses larger font) */
    .meta-table th {{ background:#fafafa; font-size:14px; }}
    .meta-table td {{ font-size:14px; }}
    .meta-empty {{ color:#666; font-style:italic; }}
  </style>
</head>
<body>
  <div class="app">
    <div class="main">
      <div class="topbar">
        <h1>Comprehensive Attack Tree</h1>
        <div class="legend">
          <span style="background:#FFD700">Root</span>
          <span style="background:#98FB98">Tactic</span>
          <span style="background:#ADD8E6">CWE</span>
          <span style="background:#90EE90">Technique</span>
          <span style="background:#FFFACD">Subtechnique</span>
          <span style="background:#FFB6C1">CAPEC</span>
          <span style="background:#FFCCCB">Gates</span>
        </div>
      </div>

      <div class="svg-wrap">
        {svg_content}
      </div>

      <p class="note">Click a node to view metadata in the right-side panel. The attack tree canvas fills the remaining screen area for easier viewing.</p>
    </div>

    <!-- placeholder for metadata panel (right column) -->
    <div id="panel-root" aria-hidden="false">
      <!-- JS will inject the metadata panel UI here -->
    </div>
  </div>

  {script_blob}
</body>
</html>
"""
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

# -------------------------
# Main run
# -------------------------
def main():
    print("Building graph...")
    dot, nodes_data, edges_data, node_metadata_map = build_attack_tree()

    # Render to SVG using Graphviz. dot.render expects filename without extension.
    basepath = os.path.join(OUTPUT_DIR, SVG_BASENAME)
    print("Rendering SVG (Graphviz). Ensure 'dot' binary available on PATH...")
    dot.render(basepath, format="svg", cleanup=True)
    svg_path = basepath + ".svg"
    print("SVG rendered at:", svg_path)

    # Inject attributes (data-node-id) and extract node positions
    print("Injecting attributes into SVG and extracting node positions...")
    node_positions = inject_tooltips_and_extract_positions(svg_path, node_metadata_map)
    print(f"Extracted positions for {len(node_positions)} nodes.")

    # Save HTML wrapper (with large canvas)
    print("Saving HTML wrapper (large canvas)...")
    save_html_with_svg(svg_path, HTML_OUTPUT, node_positions, nodes_data, node_metadata_map)
    print("HTML saved at:", HTML_OUTPUT)

    # Try PNG render for convenience
    try:
        print("Rendering PNG for convenience...")
        dot.render(basepath, format="png", cleanup=True)
        print("PNG saved at:", basepath + ".png")
    except Exception as e:
        print("PNG rendering failed (optional). Error:", e)

    print("Done. Outputs:")
    print(" - SVG:", svg_path)
    print(" - HTML:", HTML_OUTPUT)
    print(" - PNG (optional):", basepath + ".png")

if __name__ == "__main__":
    main()


Tactics (ordered): ['Initial Access', 'Persistence', 'Privilege Escalation', 'Defense Evasion', 'Credential Access', 'Discovery', 'Lateral Movement', 'Collection']
Building graph...
Rendering SVG (Graphviz). Ensure 'dot' binary available on PATH...
SVG rendered at: attack_tree_html_unified/comprehensive_attack_tree.svg
Injecting attributes into SVG and extracting node positions...
Extracted positions for 1 nodes.
Saving HTML wrapper (large canvas)...
HTML saved at: attack_tree_html_unified/comprehensive_attack_tree.html
Rendering PNG for convenience...


dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.722026 to fit


PNG saved at: attack_tree_html_unified/comprehensive_attack_tree.png
Done. Outputs:
 - SVG: attack_tree_html_unified/comprehensive_attack_tree.svg
 - HTML: attack_tree_html_unified/comprehensive_attack_tree.html
 - PNG (optional): attack_tree_html_unified/comprehensive_attack_tree.png


In [23]:
print ("hrel")        

hrel
